<div style="display: flex; gap: 20px; font-size: 0.67em;">

<div style="flex: 1;">

#### 🆔 Identity Columns
| Column Name | Description |
| :--- | :--- |
| `player` | Player's full name |
| `team` | Club name |
| `league` | League name (e.g., Premier League, La Liga) |
| `season` | Season identifier (e.g., `1718` = 2017/18 season) |
| `pos_` | Primary playing position |
| `age_` | Player's age during that season |

#### ⚽ Attacking
| Column Name | Description |
| :--- | :--- |
| `Per 90 Minutes_Gls` | Goals per 90 minutes |
| `Per 90 Minutes_Ast` | Assists per 90 minutes |
| `Per 90 Minutes_xG` | Expected goals per 90 minutes |
| `Per 90 Minutes_xAG` | Expected assisted goals per 90 minutes |

#### 🎯 Shooting
| Column Name | Description |
| :--- | :--- |
| `Standard_Sh/90` | Shots per 90 minutes |
| `Standard_SoT%` | Shot accuracy (shots on target %) |

</div>

<div style="flex: 1;">

#### 🎯 Passing
| Column Name | Description |
| :--- | :--- |
| `Total_Cmp%` | Pass completion rate (%) |
| `KP_` | Key passes |
| `PPA_` | Passes into penalty area |
| `PrgP_` | Progressive passes |

#### 🏃 Progression
| Column Name | Description |
| :--- | :--- |
| `Progression_PrgC` | Progressive carries |
| `Progression_PrgR` | Progressive receptions |

#### 🛡️ Defensive Actions
| Column Name | Description |
| :--- | :--- |
| `Tackles_Tkl` | Tackles |
| `Tackles_TklW` | Tackles won |
| `Performance_Int` | Interceptions |
| `Blocks_Blocks` | Blocks |
| `Performance_Recov` | Ball recoveries |

</div>
</div>

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from pathlib import Path

/Users/wtj/.pyenv/versions/3.11.4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download the dataset from Kaggle
dataset_path = kagglehub.dataset_download("emrey3lmaz/top-5-league-football-player-stats-2017-2025")

dataset_dir = Path(dataset_path)
csv_files = list(dataset_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV found in dataset folder.")

csv_path = csv_files[0]
print(f"Loading: {csv_path.name}")
df = pd.read_csv(csv_path, sep=';', engine='python', on_bad_lines='skip')

# Normalize header names
df.columns = df.columns.astype(str).str.strip().str.strip('"').str.strip("'")

# Make duplicate column names unique: KP_, KP_.1, KP_.2, ...
from collections import Counter
counts = Counter(df.columns)
if any(v > 1 for v in counts.values()):
    seen = {}
    new_cols = []
    for c in df.columns:
        if counts[c] > 1:
            seen[c] = seen.get(c, 0) + 1
            new_cols.append(f"{c}__dup{seen[c]}")
        else:
            new_cols.append(c)
    df.columns = new_cols
    print(f"Renamed duplicates. df now has {len(df.columns)} unique columns.")
else:
    print("No duplicate columns found.")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head(3)

Loading: Top5_League_Players_2017to2024_dataset.csv
No duplicate columns found.
Loaded 22929 rows, 178 columns


,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,...,Performance_Crs,Performance_Fld,Performance_Fls,Performance_Int,Performance_OG,Performance_Off,Performance_PKcon,Performance_PKwon,Performance_Recov,Performance_TklW
0,ENG-Premier League,1718,Arsenal,Aaron Ramsey,WAL,MF,26.0,1990.0,24,21,...,17.0,33.0,28,24.0,0.0,0.0,0.0,2.0,128.0,27.0
1,ENG-Premier League,1718,Arsenal,Ainsley Maitland-Niles,ENG,"DF,MF",19.0,1997.0,15,8,...,20.0,5.0,10,12.0,0.0,0.0,1.0,0.0,73.0,17.0
2,ENG-Premier League,1718,Arsenal,Alex Iwobi,NGA,"MF,FW",21.0,1996.0,26,22,...,20.0,12.0,11,7.0,0.0,1.0,0.0,0.0,110.0,15.0


In [3]:
# Find columns matching specific patterns
def find_cols(df, patterns):
    return sorted({c for c in df.columns if any(p.lower() in c.lower() for p in patterns)})

id_cols = find_cols(df, ['player', 'team', 'league', 'season', 'pos_', 'age_'])
id_cols = [c for c in id_cols if not c.startswith('Team Success')]
print("Final id_cols:", id_cols) # Expected: ['player', 'team', 'league', 'season', 'pos_', 'age_']

feature_cols = find_cols(df, [
    'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast',
    'Per 90 Minutes_xG', 'Per 90 Minutes_xAG',
    'Standard_Sh/90', 'Standard_SoT%',
    'Total_Cmp%', 'KP_', 'PPA_', 'PrgP_',
    'Progression_PrgC', 'Progression_PrgR',
    'Tackles_Tkl', 'Performance_Int',
    'Blocks_Blocks', 'Performance_Recov',
])
feature_cols = [c for c in feature_cols if c != 'Per 90 Minutes_xG+xAG'] # Remove Per 90 Minutes_xG+xAG, it is just xG + xAG

print(f"Features ({len(feature_cols)}):", ", ".join(repr(c) for c in feature_cols))

Final id_cols: ['age_', 'league', 'player', 'pos_', 'season', 'team']
Features (17): 'Blocks_Blocks', 'KP_', 'PPA_', 'Per 90 Minutes_Ast', 'Per 90 Minutes_Gls', 'Per 90 Minutes_xAG', 'Per 90 Minutes_xG', 'Performance_Int', 'Performance_Recov', 'PrgP_', 'Progression_PrgC', 'Progression_PrgR', 'Standard_Sh/90', 'Standard_SoT%', 'Tackles_Tkl', 'Tackles_TklW', 'Total_Cmp%'


In [4]:
# Data Cleaning
work = df.copy()
work.columns = pd.Index([str(c).strip() for c in work.columns], dtype=object)

for c in feature_cols + ['Playing Time_Min']:
    s = work[c].astype(str).str.strip()
    if s.str.contains(',', regex=False).any():
        # European format: 1.234,56 → 1234.56
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    # else: already standard (134.0), leave alone
    s = s.replace({'nan': np.nan, '': np.nan, 'None': np.nan, 'None.': np.nan})
    work[c] = pd.to_numeric(s, errors='coerce')

work = work[work['Playing Time_Min'] >= 450]

all_nan_cols = [c for c in feature_cols if work[c].isna().all()]
if all_nan_cols:
    work = work.loc[:, [c for c in work.columns if c not in all_nan_cols]]
    feature_cols = [c for c in feature_cols if c not in all_nan_cols]

for c in feature_cols:
    work[c] = work[c].fillna(0)

player_features = work[id_cols + feature_cols].copy()
print(f"✅ Player feature table: {player_features.shape}")
print(player_features.describe().T[['min', 'max', 'mean']])

✅ Player feature table: (15946, 23)
                       min      max         mean
age_                  16.0    42.00    25.940424
season              1718.0  2425.00  2072.253731
Blocks_Blocks          0.0    99.00    18.739433
KP_                    0.0   136.00    16.171140
PPA_                   0.0   140.00    13.342092
Per 90 Minutes_Ast     0.0     0.97     0.086253
Per 90 Minutes_Gls     0.0     1.50     0.123634
Per 90 Minutes_xAG     0.0     0.85     0.089214
Per 90 Minutes_xG      0.0     1.29     0.129025
Performance_Int        0.0   112.00    17.540010
Performance_Recov      0.0   345.00    89.024646
PrgP_                  0.0   424.00    66.810611
Progression_PrgC       0.0   276.00    30.773548
Progression_PrgR       0.0   674.00    65.221059
Standard_Sh/90         0.0     6.85     1.166198
Standard_SoT%          0.0   100.00    27.797749
Tackles_Tkl            0.0   152.00    28.432710
Tackles_TklW           0.0    98.00    17.266650
Total_Cmp%            34.5    95.

In [5]:
player_features.head()

,age_,league,player,pos_,season,team,Blocks_Blocks,KP_,PPA_,Per 90 Minutes_Ast,...,Performance_Int,Performance_Recov,PrgP_,Progression_PrgC,Progression_PrgR,Standard_Sh/90,Standard_SoT%,Tackles_Tkl,Tackles_TklW,Total_Cmp%
0,26.0,ENG-Premier League,Aaron Ramsey,MF,1718,Arsenal,14.0,28.0,36.0,0.39,...,24.0,128.0,134.0,61.0,161.0,2.73,44.6,35.0,27.0,83.4
1,19.0,ENG-Premier League,Ainsley Maitland-Niles,"DF,MF",1718,Arsenal,15.0,8.0,14.0,0.00,...,12.0,73.0,35.0,21.0,58.0,0.49,0.0,25.0,17.0,82.8
2,21.0,ENG-Premier League,Alex Iwobi,"MF,FW",1718,Arsenal,19.0,36.0,31.0,0.25,...,7.0,110.0,138.0,93.0,146.0,2.21,48.9,19.0,15.0,85.1
4,26.0,ENG-Premier League,Alexandre Lacazette,FW,1718,Arsenal,19.0,36.0,31.0,0.16,...,7.0,53.0,59.0,39.0,233.0,2.70,48.5,24.0,17.0,75.0
5,28.0,ENG-Premier League,Alexis Sánchez,"MF,FW",1718,Arsenal,5.0,50.0,74.0,0.18,...,6.0,48.0,149.0,50.0,218.0,4.07,35.3,20.0,9.0,69.4
